<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h1 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">تمرین: تحلیل ظرفیت سکوی تجارت الکترونیک</h1>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">در این تمرین باید یک مدل معماری برای سکوی تجارت الکترونیک بسازید، آن را به دیاگرام تبدیل کنید، دو سناریوی بار را تعریف کنید و با یک شبیه ساز ساده رفتار ظرفیت سامانه را تحلیل کنید.</p>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">هدف این است که در انتها بتوانید وضعیت روز عادی و وضعیت کمپین فروش را از نظر هزینه، بهره برداری، تاخیر و گلوگاه های اصلی با هم مقایسه کنید.</p>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed"><strong>نکته:</strong> سلول های خروجی پاک شده اند و بخش های ناقص فقط با <code>TODO</code> علامت گذاری شده اند.</p>
</div>

In [ ]:
%pip install -q "silco==0.1.3"

from collections import defaultdict, deque
from html import escape
from math import ceil
from typing import Literal

from IPython.display import HTML, SVG, display
from pydantic import BaseModel, Field

from silco import diagram


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">مدل دامنه</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">مدل ها عمداً داخل خود نوت بوک نگه داشته شده اند تا این تحلیل مستقل از کد اپلیکیشن بماند.</p>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">در این تحلیل، مدل دامنه هم ساختار معماری را توصیف می کند و هم پارامترهای لازم برای تحلیل ظرفیت و هزینه را حمل می کند.</p>
</div>

In [ ]:
NodeKind = Literal['actor', 'service', 'database', 'queue', 'cache', 'storage', 'external', 'component']


class ServiceModel(BaseModel):
    id: str
    label: str
    kind: NodeKind = 'service'
    group: str | None = None
    technology: str | None = None
    description: str | None = None
    line: int | None = None
    monthly_base_cost: float = 0.0
    cost_per_request: float = 0.0
    min_replicas: int = 1
    max_replicas: int = 20
    capacity_rps_per_replica: float = 100.0
    target_utilization: float = 0.65
    base_latency_ms: float = 8.0
    base_error_rate: float = 0.001


class LinkModel(BaseModel):
    source: str
    target: str
    protocol: str | None = None
    label: str | None = None
    mode: Literal['connect', 'flow'] = 'connect'
    traffic_share: float = 1.0
    calls_per_request: float = 1.0
    retry_rate: float = 0.0
    async_link: bool = False


class ArchitectureModel(BaseModel):
    title: str
    direction: Literal['LR', 'RL', 'TB', 'BT'] = 'RL'
    groups: dict[str, str] = Field(default_factory=dict)
    services: list[ServiceModel]
    links: list[LinkModel]


class WorkloadModel(BaseModel):
    name: str
    entry_node: str
    monthly_requests: float
    peak_rps: float
    business_priority: Literal['کم', 'متوسط', 'زیاد', 'بحرانی'] = 'متوسط'
    notes: str | None = None


class ScenarioModel(BaseModel):
    name: str
    burst_multiplier: float = 1.0
    failover_multiplier: float = 1.0
    workloads: list[WorkloadModel]


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">معماری سامانه مورد بررسی</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">سامانهٔ مورد بررسی یک سکوی تجارت الکترونیک است که مسیرهای اصلی آن شامل مرور کاتالوگ، مدیریت سبد خرید، نهایی سازی خرید، ثبت سفارش، اعلان و تحلیل داده است.</p>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">در سلول بعدی باید گروه های معماری، سرویس ها، پایگاه های داده، صف رویداد و لینک های بین آن ها را در قالب <code>ArchitectureModel</code> کامل کنید.</p>
</div>

In [ ]:
commerce = ArchitectureModel(
    title='سکوی تجارت الکترونیک',
    direction='RL',
    groups={
        # TODO
    },
    services=[
        # TODO
    ],
    links=[
        # TODO
    ],
)

raise NotImplementedError()


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">نگاشت مدل به دیاگرام</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">از روی همان مدل دامنه، نمودار معماری ساخته می شود تا تحلیل ظرفیت از نمای سیستم جدا نباشد.</p>
</div>

In [ ]:
def to_silco_diagram(model: ArchitectureModel):
    d = diagram(model.title, direction=model.direction)

    for group_id, label in model.groups.items():
        d.group(group_id, label=label)

    for service in model.services:
        metadata = {}
        if service.technology:
            metadata['technology'] = service.technology
        if service.line is not None:
            metadata['line'] = service.line

        d.node(
            service.id,
            service.label,
            kind=service.kind,
            group=service.group,
            description=service.description,
            **metadata,
        )

    for link in model.links:
        if link.mode == 'flow':
            d.flow(link.source, link.target, link.label, protocol=link.protocol)
        else:
            d.connect(link.source, link.target, link.label, protocol=link.protocol)

    return d


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">نمای معماری</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">این تصویر توپولوژی کلی سامانه و وابستگی بین سرویس های داخلی، پایگاه های داده، صف رویداد و سامانه های بیرونی را نشان می دهد.</p>
</div>

In [ ]:
commerce_diagram = to_silco_diagram(commerce)
commerce_diagram


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">سناریوهای بار کاری</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">دو وضعیت با هم مقایسه می شوند: روز عادی به عنوان وضعیت مبنا و روز کمپین به عنوان وضعیت پرریسک پیش از فروش ویژه.</p>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">در سلول بعدی باید workloadهای هر دو سناریو را طوری تعریف کنید که هم حجم ماهانه و هم اوج بار هر مسیر مشخص باشد.</p>
</div>

In [ ]:
normal_day = ScenarioModel(
    name='روز_عادی',
    burst_multiplier=1.00,
    failover_multiplier=1.00,
    workloads=[
        # TODO
    ],
)

campaign_peak = ScenarioModel(
    name='کمپین_فروش',
    burst_multiplier=1.45,
    failover_multiplier=1.15,
    workloads=[
        # TODO
    ],
)

raise NotImplementedError()


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">نمایش نتایج</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">خروجی ها به شکل شاخص کلیدی و جدول نمایش داده می شوند تا نتیجهٔ شبیه سازی مستقیماً برای بررسی معماری و برنامه ریزی ظرفیت قابل استفاده باشد.</p>
</div>

In [ ]:
def render_table(rows, columns=None, title=None):
    if not rows:
        return HTML('<div dir="rtl" lang="fa" style="color:#e5e7eb"><p>داده ای برای نمایش وجود ندارد.</p></div>')

    columns = columns or list(rows[0].keys())
    title_html = f'<h3 style="margin:0 0 12px 0;color:#f8fafc">{escape(title)}</h3>' if title else ''
    header = ''.join(
        f'<th style="text-align:right;padding:8px 10px;border-bottom:1px solid #334155;background:#e2e8f0;color:#0f172a">{escape(str(col))}</th>'
        for col in columns
    )
    body_rows = []
    for row in rows:
        cells = ''.join(
            f'<td style="padding:8px 10px;border-bottom:1px solid #e2e8f0;color:#0f172a;background:#ffffff">{escape(str(row.get(col, "")))}</td>'
            for col in columns
        )
        body_rows.append(f'<tr>{cells}</tr>')

    html = f'''
    <div dir="rtl" lang="fa" style="font-family:Tahoma, Arial, sans-serif;margin:12px 0;color:#e5e7eb;direction:rtl;text-align:right">
      {title_html}
      <div style="overflow-x:auto;border:1px solid #334155;border-radius:12px;background:#0f172a;padding:10px">
        <table style="border-collapse:collapse;min-width:960px;background:#ffffff;border-radius:8px;overflow:hidden;direction:rtl;text-align:right">
          <thead><tr>{header}</tr></thead>
          <tbody>{''.join(body_rows)}</tbody>
        </table>
      </div>
    </div>
    '''
    return HTML(html)


def render_kpis(kpis):
    items = []
    for key, value in kpis.items():
        items.append(
            f'''<div style="padding:14px 16px;border:1px solid #334155;border-radius:12px;min-width:180px;background:#f8fafc;color:#0f172a;box-shadow:0 1px 2px rgba(15,23,42,0.08)"><strong style="display:block;margin-bottom:6px;color:#334155;font-size:13px">{escape(str(key))}</strong><span style="font-size:20px;font-weight:700;color:#0f172a">{escape(str(value))}</span></div>'''
        )
    html = f'''
    <div dir="rtl" lang="fa" style="font-family:Tahoma, Arial, sans-serif;margin:12px 0;color:#e5e7eb;direction:rtl;text-align:right">
      <div style="display:flex;gap:12px;flex-wrap:wrap">{''.join(items)}</div>
    </div>
    '''
    return HTML(html)


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">موتور شبیه سازی ظرفیت</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">منطق این بخش درخواست ها را روی گراف سرویس ها پخش می کند و برای هر نود، بار، ظرفیت، تاخیر، نرخ خطا و هزینه را برآورد می کند.</p>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">در سلول بعدی باید ابتدا ترتیب توپولوژیک نودها را به دست آورید و سپس تابع <code>simulate</code> را کامل کنید تا بار ورودی روی لینک ها propagate شود و گزارش سرویس ها، هشدارها، KPIها و پرترافیک ترین edgeها تولید شوند.</p>
</div>

In [ ]:
def topological_order(model: ArchitectureModel):
    # TODO
    raise NotImplementedError()


def simulate(model: ArchitectureModel, scenario: ScenarioModel):
    # TODO
    raise NotImplementedError()


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">وضعیت مبنا در روز عادی</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">ابتدا روز عادی اجرا می شود تا خط مبنای ظرفیت، هزینه و تاخیر مشخص شود.</p>
</div>

In [ ]:
normal_result = simulate(commerce, normal_day)
display(render_kpis(normal_result['kpis']))


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">بارهای ورودی در روز عادی</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">این بخش نشان می دهد چه نوع بارهایی در وضعیت مبنا وارد سامانه می شوند.</p>
</div>

In [ ]:
display(render_table(normal_result['workloads'], title='بارهای ورودی - سناریوی عادی'))


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">وضعیت سرویس ها در روز عادی</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">در این جدول می توان دید کدام بخش ها در حالت عادی ظرفیت آزاد کافی دارند و کدام بخش ها از همین حالا به ناحیهٔ داغ نزدیک شده اند.</p>
</div>

In [ ]:
display(render_table(normal_result['services'], title='گزارش سرویس ها - سناریوی عادی'))


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">پرترافیک ترین مسیرها در روز عادی</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">این جدول برای تشخیص مسیرهایی مفید است که در وضعیت مبنا بیشترین فشار ترافیکی را حمل می کنند.</p>
</div>

In [ ]:
display(render_table(normal_result['busiest_edges'], title='پرترافیک ترین ارتباط ها - سناریوی عادی'))


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">هشدارهای وضعیت مبنا</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">اگر در روز عادی هم سرویس داغ یا بیش از ظرفیت وجود داشته باشد، باید پیش از کمپین برطرف شود.</p>
</div>

In [ ]:
display(render_table(normal_result['alerts'], title='هشدارهای ظرفیت - سناریوی عادی'))


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">سناریوی کمپین فروش ویژه</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">در این سناریو، بار لحظه ای و فشار بازیابی افزایش می یابد تا رفتار معماری در روز فشار بالا مشخص شود.</p>
</div>

In [ ]:
campaign_result = simulate(commerce, campaign_peak)
display(render_kpis(campaign_result['kpis']))


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">وضعیت سرویس ها در سناریوی کمپین</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">این جدول مهم ترین خروجی عملی نوت بوک است، چون مستقیماً نشان می دهد چه سرویس هایی برای روز کمپین ظرفیت کافی ندارند.</p>
</div>

In [ ]:
display(render_table(campaign_result['services'], title='گزارش سرویس ها - سناریوی کمپین'))


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">پرترافیک ترین مسیرها در سناریوی کمپین</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">از روی این جدول می توان دید فشار اصلی در روز کمپین روی کدام وابستگی ها و اتصال های سامانه می افتد.</p>
</div>

In [ ]:
display(render_table(campaign_result['busiest_edges'], title='پرترافیک ترین ارتباط ها - سناریوی کمپین'))


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">هشدارهای بحرانی کمپین</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">هر ردیف این جدول یک مورد عملی برای افزایش ظرفیت، بازطراحی، بازبینی راهبرد کش یا کاهش وابستگی به سامانه های بیرونی است.</p>
</div>

In [ ]:
display(render_table(campaign_result['alerts'], title='هشدارهای ظرفیت - سناریوی کمپین'))


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">مقایسهٔ وضعیت مبنا و کمپین</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">این مقایسه کمک می کند اثر واقعی کمپین روی هزینه، میزان بهره برداری و تاخیر به شکل خلاصه دیده شود.</p>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">در سلول بعدی باید یک جدول مقایسه ای از KPIهای اصلی بین دو سناریو بسازید.</p>
</div>

In [ ]:
def compare_results(normal_result, campaign_result):
    # TODO
    raise NotImplementedError()


display(render_table(compare_results(normal_result, campaign_result), title='مقایسهٔ سناریوی عادی و کمپین'))


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">
<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">جمع بندی تحلیلی</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">بر اساس خروجی سناریوی کمپین، لایهٔ وب، درگاه API، سرویس کاتالوگ، پایگاه دادهٔ کاتالوگ و کش در مسیرهایی هستند که پیش از فروش ویژه به بازبینی ظرفیت نیاز دارند.</p>
</div>